In [9]:
#Q1
import pandas as pd

roll_number = input("Enter your college roll number: ")

last_two = roll_number[-2:]
digit1 = int(last_two[0])
digit2 = int(last_two[1])

categories = ["billing", "account", "general"]

Enter your college roll number: 1024170349


In [10]:

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

personalized_entries = []

for digit in [digit1, digit2]:
    category = categories[digit % 3]

    if category == "billing":
        question = "how can i update my payment details"
        answer = "You can update your payment details through the billing section."
        keywords = "payment billing update"
    elif category == "account":
        question = "how do i update my registered mobile number"
        answer = "You can update your registered mobile number from account settings."
        keywords = "mobile number update"
    else:
        question = "what are your office timings"
        answer = "Our office is open from 9 AM to 5 PM."
        keywords = "office hours timing"

    personalized_entries.append({
        "question": question,
        "answer": answer,
        "keywords": keywords,
        "category": category
    })

all_entries = fixed_entries + personalized_entries
df = pd.DataFrame(all_entries)

print("Final 6-row DataFrame:")
display(df)

Final 6-row DataFrame:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,You can update your registered mobile number f...,mobile number update,account
5,how can i update my payment details,You can update your payment details through th...,payment billing update,billing


In [11]:
#Q2
def score_query(query, df):
    query_words = query.lower().split()
    results = []

    for _, row in df.iterrows():
        keywords = row["keywords"].lower().split()
        matched_words = [word for word in query_words if word in keywords]

        if query_words:
            confidence = len(matched_words) / len(query_words)
        else:
            confidence = 0

        if confidence > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "keywords": row["keywords"],
                "category": row["category"],
                "confidence": confidence
            })

    results.sort(key=lambda x: x["confidence"], reverse=True)
    return pd.DataFrame(results)

query = input("Enter a query: ")
result = score_query(query, df)

if result.empty:
    print("No matching entries found.")
else:
    display(result)

Enter a query: pay fees


,question,answer,keywords,category,confidence
0,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,0.5


In [12]:
#Q3
def same_category(category_name, df):
    return df[df["category"] == category_name]

personalized_category = personalized_entries[0]["category"]

print("Category:", personalized_category)
display(same_category(personalized_category, df))

Category: account


,question,answer,keywords,category
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
4,how do i update my registered mobile number,You can update your registered mobile number f...,mobile number update,account


In [13]:
#Q4
display(df[["question", "keywords"]])

entry_index = int(input("Enter the index of the entry to update (0-5): "))
new_keyword = input("Enter the new keyword: ")

df.loc[entry_index, "keywords"] = (
    df.loc[entry_index, "keywords"] + " " + new_keyword
)

filename = roll_number + "_faq_data.csv"
df.to_csv(filename, index=False)

print("Updated entry:")
display(df.loc[[entry_index]])
print("Saved as:", filename)

,question,keywords
0,what is the annual fee,fee cost price charge
1,how to reset password,password reset login
2,what are your working hours,hours timing open time
3,how can i pay the fee,pay payment upi fee
4,how do i update my registered mobile number,mobile number update
5,how can i update my payment details,payment billing update


Enter the index of the entry to update (0-5): 1
Enter the new keyword: forgot password -> change password
Updated entry:


,question,answer,keywords,category
1,how to reset password,Go to Settings > Reset Password.,password reset login forgot password -> change...,account


Saved as: 1024170349_faq_data.csv


In [14]:
#Q5
category_counts = df.groupby("category").size()
print("Number of FAQ entries per category:")
display(category_counts)

Number of FAQ entries per category:


,0
category,
account,2
billing,3
general,1


In [15]:
#Q6
def score_query_with_ties(query, df):
    query_words = query.lower().split()
    results = []

    for _, row in df.iterrows():
        keywords = row["keywords"].lower().split()
        matched_words = [word for word in query_words if word in keywords]

        if query_words:
            confidence = len(matched_words) / len(query_words)
        else:
            confidence = 0

        if confidence > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "keywords": row["keywords"],
                "category": row["category"],
                "confidence": confidence
            })

    if not results:
        return pd.DataFrame()

    result_df = pd.DataFrame(results)
    highest_score = result_df["confidence"].max()

    return result_df[result_df["confidence"] == highest_score]

# Test 1: query that can produce a tie
tie_query = "fee"
print("Tie test query:", tie_query)
tie_result = score_query_with_ties(tie_query, df)
display(tie_result)

# Test 2: query that produces no match
no_match_query = "elephant"
print("No-match test query:", no_match_query)
no_match_result = score_query_with_ties(no_match_query, df)

if no_match_result.empty:
    print("No matching entries found.")
else:
    display(no_match_result)

Tie test query: fee


,question,answer,keywords,category,confidence
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing,1.0
1,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,1.0


No-match test query: elephant
No matching entries found.
